In [ ]:
# 统一读取路径与数据

In [5]:
!pip install numpy pandas matplotlib

In [6]:
# ==========================================
# 0. 本地运行配置：项目路径、结果路径、读取数据
# ==========================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 修改为你的本地项目目录
# 你的截图路径：
# C:\Users\len\Desktop\算电协同\HMM-LSTM\data
PROJECT_DIR = Path(r"C:\Users\len\Desktop\算电协同\HMM-LSTM")

RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "diagnostic_figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# 优先读取实验生成的数据
DATA_PATH = RESULTS_DIR / "merged_hourly_data.csv"

# 如果没有results文件夹里的数据，则读取你截图中的原始csv
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_DIR / "data" / "17-25.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"没有找到数据文件，请检查路径：{DATA_PATH}"
    )

print("正在读取：", DATA_PATH)

df = pd.read_csv(DATA_PATH)

# 时间字段转换
if "timestamp" not in df.columns:
    raise KeyError(
        "数据中没有 timestamp 列，请确认CSV是否为处理后的小时级数据。"
    )

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

TARGET_COL = "carbon_intensity_gCO2eq_per_kWh"

if TARGET_COL not in df.columns:
    raise KeyError(
        f"数据中没有目标列 {TARGET_COL}，请检查CSV字段。"
    )

print("数据读取成功")
print("数据量：", df.shape)
print("字段数量：", len(df.columns))


FileNotFoundError: 没有找到数据文件，请检查路径：C:\Users\len\Desktop\算电协同\HMM-LSTM/data/17-25.csv

In [ ]:
# 季节性强度

In [ ]:
# ==========================================
# 1. 检查季节性强度
# ==========================================

series = pd.to_numeric(df[TARGET_COL], errors="coerce").dropna()

lags_to_check = [1, 24, 48, 168]

autocorr_rows = []

for lag in lags_to_check:
    autocorr_value = series.autocorr(lag=lag)

    # 计算简单滞后预测误差：
    # 用 t-lag 时刻的值预测 t 时刻
    actual = series.iloc[lag:].to_numpy()
    lag_pred = series.shift(lag).iloc[lag:].to_numpy()

    mae = np.mean(np.abs(actual - lag_pred))
    rmse = np.sqrt(np.mean((actual - lag_pred) ** 2))

    autocorr_rows.append({
        "lag_hours": lag,
        "autocorrelation": autocorr_value,
        "lag_naive_MAE": mae,
        "lag_naive_RMSE": rmse
    })

seasonality_table = pd.DataFrame(autocorr_rows)

print("关键滞后自相关与简单滞后预测误差：")
display(seasonality_table.round(4))

# 绘制 1 至 336 小时自相关
max_lag = 336
acf_values = [
    series.autocorr(lag=lag)
    for lag in range(1, max_lag + 1)
]

plt.figure(figsize=(14, 5))
plt.plot(range(1, max_lag + 1), acf_values)

for lag in [24, 48, 168, 336]:
    if lag <= max_lag:
        plt.axvline(lag, linestyle="--", alpha=0.7)
        plt.text(
            lag,
            max(acf_values),
            f"{lag}h",
            rotation=90,
            verticalalignment="top"
        )

plt.xlabel("滞后小时数")
plt.ylabel("自相关系数")
plt.title("碳强度时间序列自相关：1至336小时")
plt.tight_layout()

seasonality_fig_path = FIGURES_DIR / "seasonality_autocorrelation.png"
plt.savefig(seasonality_fig_path, dpi=180)
plt.show()

# 简单判断
corr_24 = seasonality_table.loc[
    seasonality_table["lag_hours"] == 24,
    "autocorrelation"
].iloc[0]

corr_168 = seasonality_table.loc[
    seasonality_table["lag_hours"] == 168,
    "autocorrelation"
].iloc[0]

print("\n自动判断：")

if corr_24 >= 0.7:
    print(f"24小时周期很强，自相关为 {corr_24:.3f}。")
elif corr_24 >= 0.4:
    print(f"24小时周期中等，自相关为 {corr_24:.3f}。")
else:
    print(f"24小时周期较弱，自相关为 {corr_24:.3f}。")

if corr_168 >= 0.7:
    print(f"168小时周周期很强，自相关为 {corr_168:.3f}。")
elif corr_168 >= 0.4:
    print(f"168小时周周期中等，自相关为 {corr_168:.3f}。")
else:
    print(f"168小时周周期较弱，自相关为 {corr_168:.3f}。")

seasonality_table.to_csv(
    RESULTS_DIR / "diagnostic_seasonality.csv",
    index=False
)

print("\n已保存：")
print(RESULTS_DIR / "diagnostic_seasonality.csv")
print(seasonality_fig_path)

In [ ]:
# HMM状态样本量与持续时间

In [ ]:
# ==========================================
# 2. 检查 HMM 各状态样本量和持续时间
# ==========================================

HMM_SUMMARY_PATH = RESULTS_DIR / "hmm_state_summary.csv"

if HMM_SUMMARY_PATH.exists():
    hmm_summary = pd.read_csv(HMM_SUMMARY_PATH)
    print("读取已有 HMM 状态汇总文件：")
    display(hmm_summary.round(4))
else:
    print("未找到 hmm_state_summary.csv，将从 merged_hourly_data.csv 重新统计。")

    if "hmm_state" not in df.columns:
        raise KeyError(
            "数据中没有 hmm_state 列，且 results/hmm_state_summary.csv 不存在。"
        )

    # 重新计算状态持续时间
    state_change_group = (
        df["hmm_state"] != df["hmm_state"].shift()
    ).cumsum()

    temp = df.copy()
    temp["recomputed_state_duration"] = (
        temp.groupby(state_change_group).cumcount() + 1
    )

    hmm_summary = (
        temp.groupby("hmm_state")
        .agg(
            sample_count=("hmm_state", "size"),
            carbon_mean=(TARGET_COL, "mean"),
            carbon_std=(TARGET_COL, "std"),
            mean_duration=("recomputed_state_duration", "mean"),
            max_duration=("recomputed_state_duration", "max")
        )
        .reset_index()
    )

    hmm_summary["sample_share"] = (
        hmm_summary["sample_count"] / len(temp)
    )

# 如果原文件没有 sample_count，则补充
if "sample_count" not in hmm_summary.columns:
    if "hmm_state" not in df.columns:
        raise KeyError("完整数据中没有 hmm_state 列。")

    state_counts = (
        df["hmm_state"]
        .value_counts()
        .sort_index()
        .rename("sample_count")
        .reset_index()
        .rename(columns={"index": "hmm_state"})
    )

    hmm_summary = hmm_summary.merge(
        state_counts,
        on="hmm_state",
        how="left"
    )

# 排序
hmm_summary = hmm_summary.sort_values("hmm_state").reset_index(drop=True)

print("\nHMM状态详细统计：")
display(hmm_summary.round(4))

# 绘制样本占比
plt.figure(figsize=(8, 5))
plt.bar(
    hmm_summary["hmm_state"].astype(str),
    hmm_summary["sample_share"]
)

plt.xlabel("HMM状态")
plt.ylabel("样本占比")
plt.title("HMM各状态样本占比")
plt.tight_layout()

state_share_fig_path = FIGURES_DIR / "hmm_state_sample_share.png"
plt.savefig(state_share_fig_path, dpi=180)
plt.show()

# 绘制平均持续时间
if "mean_duration" in hmm_summary.columns:
    plt.figure(figsize=(8, 5))
    plt.bar(
        hmm_summary["hmm_state"].astype(str),
        hmm_summary["mean_duration"]
    )

    plt.xlabel("HMM状态")
    plt.ylabel("平均持续时间 / 小时")
    plt.title("HMM各状态平均持续时间")
    plt.tight_layout()

    duration_fig_path = FIGURES_DIR / "hmm_state_mean_duration.png"
    plt.savefig(duration_fig_path, dpi=180)
    plt.show()

# 自动风险判断
print("\n自动诊断：")

for _, row in hmm_summary.iterrows():
    state = int(row["hmm_state"])
    share = float(row["sample_share"])

    if share < 0.05:
        share_level = "严重不足"
    elif share < 0.10:
        share_level = "偏少"
    elif share < 0.20:
        share_level = "一般偏少"
    else:
        share_level = "较充足"

    print(
        f"状态 {state}：样本占比 {share:.2%}，"
        f"样本量评价：{share_level}"
    )

    if "mean_duration" in row and pd.notna(row["mean_duration"]):
        duration = float(row["mean_duration"])

        if duration < 2:
            print(
                f"  平均持续时间仅 {duration:.2f} 小时，"
                "状态切换非常频繁，不利于状态专家LSTM。"
            )
        elif duration < 6:
            print(
                f"  平均持续时间为 {duration:.2f} 小时，"
                "状态稳定性偏弱。"
            )
        else:
            print(
                f"  平均持续时间为 {duration:.2f} 小时，"
                "状态具有一定稳定性。"
            )

hmm_summary.to_csv(
    RESULTS_DIR / "diagnostic_hmm_state_balance.csv",
    index=False
)

print("\n已保存：")
print(RESULTS_DIR / "diagnostic_hmm_state_balance.csv")
print(state_share_fig_path)

In [ ]:
# 检验HMM状态对1、6、12、24的解释程度

In [ ]:
# ==========================================
# 3. 检查 HMM 状态与未来目标的关系
# ==========================================

if "hmm_state" not in df.columns:
    raise KeyError("merged_hourly_data.csv 中没有 hmm_state 列。")

# 尝试读取训练集计算出的高碳阈值
METADATA_PATH = RESULTS_DIR / "experiment_metadata.json"

high_threshold = None

if METADATA_PATH.exists():
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    high_threshold = (
        metadata
        .get("target", {})
        .get("high_threshold")
    )

if high_threshold is None:
    # 兜底方案：
    # 这里只用于诊断，不建议用于正式论文结果
    high_threshold = df[TARGET_COL].quantile(0.80)
    print(
        "警告：没有从 experiment_metadata.json 读取到训练集阈值，"
        "暂时使用全样本80%分位数。"
    )

high_threshold = float(high_threshold)

print("高碳阈值：", high_threshold)

future_df = df[
    ["timestamp", TARGET_COL, "hmm_state"]
].copy()

# 未来第1小时
future_df["future_1h"] = future_df[TARGET_COL].shift(-1)

# 未来1至6小时均值
future_df["future_6h_mean"] = pd.concat(
    [
        future_df[TARGET_COL].shift(-h)
        for h in range(1, 7)
    ],
    axis=1
).mean(axis=1)

# 未来1至12小时均值
future_df["future_12h_mean"] = pd.concat(
    [
        future_df[TARGET_COL].shift(-h)
        for h in range(1, 13)
    ],
    axis=1
).mean(axis=1)

# 未来1至24小时均值
future_24_matrix = pd.concat(
    [
        future_df[TARGET_COL].shift(-h)
        for h in range(1, 25)
    ],
    axis=1
)

future_df["future_24h_mean"] = future_24_matrix.mean(axis=1)

# 未来24小时内是否至少出现一次高碳
future_df["future_24h_high_event"] = (
    future_24_matrix.gt(high_threshold).any(axis=1).astype(float)
)

# 最后24行没有完整未来窗口，删除
future_df = future_df.iloc[:-24].copy()

future_relation = (
    future_df.groupby("hmm_state")
    .agg(
        sample_count=("hmm_state", "size"),
        current_carbon_mean=(TARGET_COL, "mean"),
        future_1h_mean=("future_1h", "mean"),
        future_6h_mean=("future_6h_mean", "mean"),
        future_12h_mean=("future_12h_mean", "mean"),
        future_24h_mean=("future_24h_mean", "mean"),
        future_24h_high_rate=("future_24h_high_event", "mean")
    )
    .reset_index()
    .sort_values("hmm_state")
)

print("不同HMM状态对应的未来目标统计：")
display(future_relation.round(4))

# 计算状态间差异范围
difference_summary = pd.DataFrame({
    "future_horizon": [
        "1h",
        "6h",
        "12h",
        "24h",
        "24h_high_event"
    ],
    "max_minus_min": [
        future_relation["future_1h_mean"].max()
        - future_relation["future_1h_mean"].min(),

        future_relation["future_6h_mean"].max()
        - future_relation["future_6h_mean"].min(),

        future_relation["future_12h_mean"].max()
        - future_relation["future_12h_mean"].min(),

        future_relation["future_24h_mean"].max()
        - future_relation["future_24h_mean"].min(),

        future_relation["future_24h_high_rate"].max()
        - future_relation["future_24h_high_rate"].min()
    ]
})

print("\n不同状态之间的最大差异：")
display(difference_summary.round(4))

# 计算 eta squared：HMM状态对目标方差的解释比例
def eta_squared(data, group_col, value_col):
    valid = data[[group_col, value_col]].dropna()

    overall_mean = valid[value_col].mean()

    ss_between = sum(
        len(group) * (group[value_col].mean() - overall_mean) ** 2
        for _, group in valid.groupby(group_col)
    )

    ss_total = (
        (valid[value_col] - overall_mean) ** 2
    ).sum()

    if ss_total == 0:
        return np.nan

    return ss_between / ss_total


effect_rows = []

for col in [
    "future_1h",
    "future_6h_mean",
    "future_12h_mean",
    "future_24h_mean",
    "future_24h_high_event"
]:
    effect_rows.append({
        "target": col,
        "eta_squared": eta_squared(
            future_df,
            "hmm_state",
            col
        )
    })

effect_table = pd.DataFrame(effect_rows)

print("\nHMM状态解释力 eta squared：")
display(effect_table.round(4))

# 绘图：不同状态的未来均值
plot_data = future_relation.set_index("hmm_state")[
    [
        "future_1h_mean",
        "future_6h_mean",
        "future_12h_mean",
        "future_24h_mean"
    ]
]

plt.figure(figsize=(10, 6))

for column in plot_data.columns:
    plt.plot(
        plot_data.index,
        plot_data[column],
        marker="o",
        label=column
    )

plt.xlabel("当前HMM状态")
plt.ylabel("未来碳强度均值")
plt.title("不同HMM状态下的未来碳强度")
plt.legend()
plt.tight_layout()

future_relation_fig_path = (
    FIGURES_DIR / "hmm_state_future_relationship.png"
)

plt.savefig(future_relation_fig_path, dpi=180)
plt.show()

# 绘制未来高碳风险
plt.figure(figsize=(8, 5))
plt.bar(
    future_relation["hmm_state"].astype(str),
    future_relation["future_24h_high_rate"]
)

plt.xlabel("当前HMM状态")
plt.ylabel("未来24小时高碳事件比例")
plt.title("不同HMM状态下未来24小时高碳风险")
plt.tight_layout()

future_risk_fig_path = (
    FIGURES_DIR / "hmm_state_future_high_risk.png"
)

plt.savefig(future_risk_fig_path, dpi=180)
plt.show()

future_relation.to_csv(
    RESULTS_DIR / "diagnostic_hmm_future_relationship.csv",
    index=False
)

effect_table.to_csv(
    RESULTS_DIR / "diagnostic_hmm_effect_size.csv",
    index=False
)

print("\n已保存：")
print(RESULTS_DIR / "diagnostic_hmm_future_relationship.csv")
print(RESULTS_DIR / "diagnostic_hmm_effect_size.csv")

In [ ]:
# 可再生能源及其他输入特征质量

In [ ]:
# ==========================================
# 4. 检查可再生能源特征质量
# ==========================================

# 识别可能与可再生能源有关的列
renewable_keywords = [
    "renewable",
    "solar",
    "wind",
    "hydro",
    "biomass",
    "geothermal"
]

renewable_cols = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in renewable_keywords)
]

# 排除明显不是数值特征的字段
renewable_numeric_cols = [
    col for col in renewable_cols
    if pd.api.types.is_numeric_dtype(df[col])
]

print("识别出的可再生能源相关数值列：")
for col in renewable_numeric_cols:
    print(" -", col)

# 也检查所有数值列
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

quality_rows = []

for col in numeric_cols:
    values = pd.to_numeric(df[col], errors="coerce")

    missing_rate = values.isna().mean()
    zero_rate = (values == 0).mean()
    unique_count = values.nunique(dropna=True)
    std_value = values.std()
    mean_value = values.mean()
    min_value = values.min()
    max_value = values.max()

    # 近恒定判断：最常见值占比
    value_counts = values.value_counts(
        normalize=True,
        dropna=False
    )

    dominant_rate = (
        value_counts.iloc[0]
        if len(value_counts) > 0
        else np.nan
    )

    quality_rows.append({
        "feature": col,
        "missing_rate": missing_rate,
        "zero_rate": zero_rate,
        "unique_count": unique_count,
        "mean": mean_value,
        "std": std_value,
        "min": min_value,
        "max": max_value,
        "dominant_value_rate": dominant_rate,
        "is_constant": unique_count <= 1,
        "is_near_constant": dominant_rate >= 0.99
    })

quality_table = pd.DataFrame(quality_rows)

# 与目标变量的相关性
corr_series = (
    df[numeric_cols]
    .corr(method="pearson")[TARGET_COL]
    .rename("pearson_corr_with_target")
)

quality_table = quality_table.merge(
    corr_series,
    left_on="feature",
    right_index=True,
    how="left"
)

quality_table = quality_table.sort_values(
    ["missing_rate", "is_near_constant"],
    ascending=[False, False]
)

print("\n所有数值特征质量汇总：")
display(quality_table.round(4))

# 仅查看可再生能源相关列
renewable_quality = quality_table[
    quality_table["feature"].isin(renewable_numeric_cols)
].copy()

print("\n可再生能源相关特征质量：")
display(renewable_quality.round(4))

# 时间连续性检查
time_diff = df["timestamp"].diff().dropna()

expected_diff = pd.Timedelta(hours=1)

irregular_intervals = time_diff[time_diff != expected_diff]

print("\n时间对齐检查：")
print("总时间间隔数量：", len(time_diff))
print("非1小时间隔数量：", len(irregular_intervals))

if len(irregular_intervals) > 0:
    print("\n异常时间间隔示例：")
    display(
        pd.DataFrame({
            "timestamp": df.loc[
                irregular_intervals.index,
                "timestamp"
            ],
            "time_difference": irregular_intervals
        }).head(20)
    )
else:
    print("时间序列严格保持1小时间隔。")

# 缺失后是否可能被统一填0
print("\n疑似统一填0的特征：")

suspected_zero_filled = quality_table[
    (quality_table["zero_rate"] > 0.50)
    & (quality_table["unique_count"] > 1)
][
    [
        "feature",
        "zero_rate",
        "missing_rate",
        "unique_count",
        "std"
    ]
]

display(suspected_zero_filled.round(4))

# 恒定和近恒定列
constant_features = quality_table[
    quality_table["is_constant"]
]

near_constant_features = quality_table[
    quality_table["is_near_constant"]
]

print("\n恒定列：")
display(
    constant_features[
        [
            "feature",
            "unique_count",
            "dominant_value_rate"
        ]
    ]
)

print("\n近恒定列：")
display(
    near_constant_features[
        [
            "feature",
            "unique_count",
            "dominant_value_rate"
        ]
    ]
)

# 可再生能源特征内部相关性
if len(renewable_numeric_cols) >= 2:
    renewable_corr = df[renewable_numeric_cols].corr()

    print("\n可再生能源特征相关矩阵：")
    display(renewable_corr.round(3))

    plt.figure(figsize=(10, 8))
    plt.imshow(
        renewable_corr,
        aspect="auto",
        interpolation="nearest"
    )

    plt.xticks(
        range(len(renewable_corr.columns)),
        renewable_corr.columns,
        rotation=90
    )

    plt.yticks(
        range(len(renewable_corr.index)),
        renewable_corr.index
    )

    plt.colorbar(label="相关系数")
    plt.title("可再生能源特征相关矩阵")
    plt.tight_layout()

    renewable_corr_fig_path = (
        FIGURES_DIR / "renewable_feature_correlation.png"
    )

    plt.savefig(renewable_corr_fig_path, dpi=180)
    plt.show()

# 检查 renewable_percentage 是否可能由其他列直接计算
if "renewable_percentage" in df.columns:
    candidate_generation_cols = [
        col for col in renewable_numeric_cols
        if col != "renewable_percentage"
    ]

    print("\nrenewable_percentage 与其他可再生能源列的相关性：")

    renewable_percentage_corr = (
        df[
            ["renewable_percentage"]
            + candidate_generation_cols
        ]
        .corr()["renewable_percentage"]
        .drop("renewable_percentage")
        .sort_values(
            key=lambda x: x.abs(),
            ascending=False
        )
    )

    display(
        renewable_percentage_corr
        .rename("correlation")
        .to_frame()
        .round(4)
    )

    very_high_corr = renewable_percentage_corr[
        renewable_percentage_corr.abs() >= 0.98
    ]

    if len(very_high_corr) > 0:
        print(
            "警告：renewable_percentage 与以下特征相关性绝对值达到0.98以上，"
            "可能存在重复信息或近似机械计算关系："
        )
        display(very_high_corr)
    else:
        print(
            "没有发现 renewable_percentage 与单一特征之间"
            "绝对相关性达到0.98以上。"
        )

# 自动输出问题清单
issues = []

for _, row in quality_table.iterrows():
    feature = row["feature"]

    if row["missing_rate"] > 0.05:
        issues.append({
            "feature": feature,
            "issue": "missing_rate_above_5_percent",
            "value": row["missing_rate"]
        })

    if row["zero_rate"] > 0.80 and row["unique_count"] > 1:
        issues.append({
            "feature": feature,
            "issue": "zero_rate_above_80_percent",
            "value": row["zero_rate"]
        })

    if row["is_constant"]:
        issues.append({
            "feature": feature,
            "issue": "constant_feature",
            "value": row["unique_count"]
        })

    elif row["is_near_constant"]:
        issues.append({
            "feature": feature,
            "issue": "near_constant_feature",
            "value": row["dominant_value_rate"]
        })

issues_table = pd.DataFrame(issues)

print("\n自动识别的问题特征：")
display(issues_table)

quality_table.to_csv(
    RESULTS_DIR / "diagnostic_feature_quality.csv",
    index=False
)

renewable_quality.to_csv(
    RESULTS_DIR / "diagnostic_renewable_quality.csv",
    index=False
)

issues_table.to_csv(
    RESULTS_DIR / "diagnostic_feature_issues.csv",
    index=False
)

print("\n已保存：")
print(RESULTS_DIR / "diagnostic_feature_quality.csv")
print(RESULTS_DIR / "diagnostic_renewable_quality.csv")
print(RESULTS_DIR / "diagnostic_feature_issues.csv")

In [ ]:
# 诊断报告

In [ ]:
# ==========================================
# 5. 汇总四项诊断结果
# ==========================================

print("=" * 60)
print("四项诊断汇总")
print("=" * 60)

# 1. 季节性
seasonality_result = pd.read_csv(
    RESULTS_DIR / "diagnostic_seasonality.csv"
)

corr_24 = seasonality_result.loc[
    seasonality_result["lag_hours"] == 24,
    "autocorrelation"
].iloc[0]

corr_168 = seasonality_result.loc[
    seasonality_result["lag_hours"] == 168,
    "autocorrelation"
].iloc[0]

print("\n1. 季节性")
print(f"24小时自相关：{corr_24:.4f}")
print(f"168小时自相关：{corr_168:.4f}")

if max(corr_24, corr_168) >= 0.7:
    print("结论：周期性很强，B1季节残差表现好具有明确数据依据。")
elif max(corr_24, corr_168) >= 0.4:
    print("结论：周期性中等，季节残差仍有合理性。")
else:
    print("结论：周期性不强，需要重新检查B1优势来源。")

# 2. 状态平衡
state_balance = pd.read_csv(
    RESULTS_DIR / "diagnostic_hmm_state_balance.csv"
)

min_share = state_balance["sample_share"].min()

print("\n2. HMM状态平衡")
print(f"最小状态样本占比：{min_share:.2%}")

if min_share < 0.05:
    print("结论：存在严重少数状态，不适合使用多个状态专属LSTM专家。")
elif min_share < 0.10:
    print("结论：部分状态样本偏少，状态专家模型风险较高。")
else:
    print("结论：各状态样本量没有出现严重失衡。")

# 3. HMM未来解释力
effect_table = pd.read_csv(
    RESULTS_DIR / "diagnostic_hmm_effect_size.csv"
)

print("\n3. HMM对未来目标的解释力")
display(effect_table.round(4))

eta_1h = effect_table.loc[
    effect_table["target"] == "future_1h",
    "eta_squared"
].iloc[0]

eta_24h = effect_table.loc[
    effect_table["target"] == "future_24h_mean",
    "eta_squared"
].iloc[0]

eta_risk = effect_table.loc[
    effect_table["target"] == "future_24h_high_event",
    "eta_squared"
].iloc[0]

if eta_1h > eta_24h * 2 and eta_1h >= 0.06:
    print(
        "结论：HMM更适合短期修正，"
        "不适合强控制整个24小时轨迹。"
    )
elif eta_24h >= 0.06:
    print(
        "结论：HMM对24小时平均目标有一定解释力，"
        "可以尝试弱门控。"
    )
else:
    print(
        "结论：HMM对24小时连续预测解释力较弱，"
        "不建议继续使用强状态混合。"
    )

if eta_risk > eta_24h:
    print(
        "补充结论：HMM对风险分类的价值高于对连续回归的价值。"
    )

# 4. 特征质量
feature_issues = pd.read_csv(
    RESULTS_DIR / "diagnostic_feature_issues.csv"
)

print("\n4. 特征质量")
print("自动识别问题数量：", len(feature_issues))

if len(feature_issues) == 0:
    print("结论：未发现明显缺失、恒定或极高零值比例问题。")
else:
    issue_counts = feature_issues["issue"].value_counts()
    display(issue_counts.rename("count").to_frame())

    print(
        "结论：应先处理问题特征，再判断复杂神经网络结构是否有效。"
    )

print("\n" + "=" * 60)
print("建议判断")
print("=" * 60)

data_problem_score = 0
model_problem_score = 0

if len(feature_issues) > 0:
    data_problem_score += 1

if min_share < 0.10:
    model_problem_score += 1

if eta_24h < 0.06:
    model_problem_score += 1

if max(corr_24, corr_168) >= 0.7:
    model_problem_score += 1

print("数据问题信号：", data_problem_score)
print("模型结构问题信号：", model_problem_score)

if data_problem_score > model_problem_score:
    print(
        "总体建议：先处理数据质量，"
        "尤其是缺失、零值、恒定列和时间对齐问题。"
    )
elif model_problem_score > data_problem_score:
    print(
        "总体建议：当前更像是模型结构问题。"
        "优先停止B9强状态混合，改为B10弱残差门控。"
    )
else:
    print(
        "总体建议：数据和模型都需要处理。"
        "先清理明显问题特征，再测试B10弱残差门控。"
    )